In [4]:
import pandas as pd
import requests 
import numpy as np
import requests
import re

In [ ]:
# first pass at abstracts for all journals 

df = pd.read_excel("Public version OpenAccess Dataset.xlsx", engine="openpyxl")

def pubmed_search(title, journal, year, volume=None, issue=None, page=None):
    terms = [
        f'({title}[ti])',
        f'({journal}[jour])',
        f'({year}[dp])'
    ]
    if volume:
        terms.append(f'({volume}[vi])')
    if issue:
        terms.append(f'({issue}[ip])')
    if page:
        terms.append(f'({page}[pg])')
        
    query = " AND ".join(terms)

    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pubmed", "term": query, "retmode": "json"}
    r = requests.get(url, params=params).json()

    ids = r["esearchresult"]["idlist"]
    return ids[0] if ids else None

def pubmed_search2(journal, year, volume=None, issue=None, page=None):
    terms = [
        f'({journal}[jour])',
        f'({year}[dp])'
    ]
    if volume:
        terms.append(f'({volume}[vi])')
    if issue:
        terms.append(f'({issue}[ip])')
    if page:
        terms.append(f'({page}[pg])')
        
    query = " AND ".join(terms)

    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pubmed", "term": query, "retmode": "json"}
    r = requests.get(url, params=params).json()

    ids = r["esearchresult"]["idlist"]
    return ids[0] if ids else None

def get_pubmed_abstract(pmid):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {"db": "pubmed", "id": pmid, "retmode": "text", "rettype": "abstract"}
    return requests.get(url, params=params).text

import time
for i in range(0, len(df)):
    try: 
        pmid = pubmed_search(
         title=df['Article Title'].iloc[i],
         journal=df['Journal'].iloc[i],
         year=df['Year'].iloc[i],
         volume=df['Volume'].iloc[i],
         issue=df['Issue'].iloc[i],
         page=df['Beginning page'].iloc[i])
        if pmid:
            df.at[i, 'extra_info'] = get_pubmed_abstract(pmid)
        else:
            df.at[i, 'extra_info'] = np.nan
    except Exception as e:
        print(i)
        print(f"An unexpected error occurred: {e}")
    time.sleep(1)

# rerun for ones where this didn't work 
for i in range(0, len(df)):
    if pd.isna(df['extra_info'].iloc[i]):
        try: 
            pmid = pubmed_search2(
            journal=df['Journal'].iloc[i],
            year=df['Year'].iloc[i],
            volume=df['Volume'].iloc[i],
            issue=df['Issue'].iloc[i],
            page=df['Beginning page'].iloc[i])
            if pmid:
                df.at[i, 'extra_info'] = get_pubmed_abstract(pmid)
            else:
                df.at[i, 'extra_info'] = np.nan
        except Exception as e:
            print(i)
            print(f"An unexpected error occurred: {e}")
        time.sleep(0.35)

df.to_csv("open_access_data_with_abstracts.csv")


In [ ]:
# SCIENCE
df = pd.read_csv("open_access_data_with_abstracts.csv")

science = df[df['Journal'] == "Science"].reset_index(drop = True).copy()
for i in range(0, len(science)):
    if science['title'].iloc[i] is None:
        print(i)
        try: 
            pmid = pubmed_search2(
            journal=science['Journal'].iloc[i],
            year=science['Year'].iloc[i],
            volume=science['Volume'].iloc[i],
            issue=science['Issue'].iloc[i],
            page=science['Beginning page'].iloc[i])
            if pmid:
                science.at[i, 'extra_info'] = get_pubmed_abstract(pmid)
            else:
                science.at[i, 'extra_info'] = np.nan
        except Exception as e:
            print(i)
            print(f"An unexpected error occurred: {e}")
        time.sleep(1)


def count_paragraphs(text, n=6):
    if not isinstance(text, str):
        return [None] * n
    paragraphs = re.split(r'\n\s*\n', text)
    
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    return len(paragraphs)

science['num_paragraphs'] = science['extra_info'].apply(count_paragraphs)

def split_and_fix_paragraphs(text):
    # Handle NaN, None, non-strings
    if not isinstance(text, str):
        return ["" for _ in range(7)]

    # Split on blank lines (paragraph separators)
    paragraphs = re.split(r'\n\s*\n', text)
    
    # Strip whitespace around each paragraph
    paragraphs = [p.strip() for p in paragraphs if p.strip() != ""]

    # ---------- Apply your rules ----------

    # Case 1: Exactly 7 paragraphs → leave as is
    if len(paragraphs) == 7:
        return paragraphs

    # Case 2: Exactly 6 paragraphs → insert empty string at index 4 (5th position)
    if len(paragraphs) == 6:
        paragraphs.insert(4, "")
        return paragraphs

    # Case 3: Exactly 8 paragraphs → combine paragraphs 6 and 7
    if len(paragraphs) == 8:
        combined = paragraphs[4] + " " + paragraphs[5]
        fixed = paragraphs[:4] + [combined] + [paragraphs[6]]
        return fixed  # now length 7

    # Everything else: normalize to length 7 (pad or trim)
    if len(paragraphs) < 7:
        return paragraphs + [""] * (7 - len(paragraphs))
    else:
        return paragraphs[:7]

science[['journal_info','title','authors','author_info', 'extra_comments', 'abstract','doi']] = science['extra_info'].apply(split_and_fix_paragraphs).apply(pd.Series)
science.to_csv("science_openaccess_journal_data/science.csv", index = False)

In [ ]:
# functions for four later journals

def split_paragraphs(text, n=6):
    if not isinstance(text, str):
        return [None] * n
    paragraphs = re.split(r'\n\s*\n', text)
    
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    while len(paragraphs) < n:
        paragraphs.append(None)
    
    return paragraphs[:n]
def pubmed_search2(journal, year, volume=None, issue=None, page=None):
    terms = [
        f'({journal}[jour])',
        f'({year}[dp])'
    ]
    if volume:
        terms.append(f'({volume}[vi])')
    if issue:
        terms.append(f'({issue}[ip])')
    if page:
        terms.append(f'({page}[pg])')
        
    query = " AND ".join(terms)

    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pubmed", "term": query, "retmode": "json"}
    r = requests.get(url, params=params).json()

    ids = r["esearchresult"]["idlist"]
    return ids[0] if ids else None

def get_pubmed_abstract(pmid):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {"db": "pubmed", "id": pmid, "retmode": "text", "rettype": "abstract"}
    return requests.get(url, params=params).text


def split_and_fix_paragraphs(text):
    # Handle NaN, None, non-strings
    if not isinstance(text, str):
        return ["" for _ in range(7)]

    # Split on blank lines (paragraph separators)
    paragraphs = re.split(r'\n\s*\n', text)
    
    # Strip whitespace around each paragraph
    paragraphs = [p.strip() for p in paragraphs if p.strip() != ""]

    # ---------- Apply your rules ----------

    # Case 1: Exactly 7 paragraphs → leave as is
    if len(paragraphs) == 7:
        return paragraphs

    # Case 2: Exactly 6 paragraphs → insert empty string at index 4 (5th position)
    if len(paragraphs) == 6:
        paragraphs.insert(4, "")
        return paragraphs

    # Case 3: Exactly 8 paragraphs → combine paragraphs 6 and 7
    if len(paragraphs) == 8:
        combined = paragraphs[4] + " " + paragraphs[5]
        fixed = paragraphs[:4] + [combined] + [paragraphs[6]]
        return fixed  # now length 7

    # Everything else: normalize to length 7 (pad or trim)
    if len(paragraphs) < 7:
        return paragraphs + [""] * (7 - len(paragraphs))
    else:
        return paragraphs[:7]
    
def count_paragraphs(text, n=6):
    if not isinstance(text, str):
        return [None] * n
    paragraphs = re.split(r'\n\s*\n', text)
    
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    return len(paragraphs)

In [ ]:
# NEURO
df = pd.read_csv("open_access_data_with_abstracts.csv")

neuro = df[df['Journal'] == "J. Neurophysiol."].reset_index(drop = True).copy()

neuro[['journal_info','title','authors','author_info','abstract','doi']] = neuro['extra_info'].apply(split_paragraphs).apply(pd.Series)

import time

for i in range(0, len(neuro )):
    if neuro ['title'].iloc[i] is None:
        print(i)
        try: 
            pmid = pubmed_search2(
            journal=neuro['Journal'].iloc[i],
            year= neuro['Year'].iloc[i],
            volume= neuro['Volume'].iloc[i],
            issue= neuro['Issue'].iloc[i],
            page= neuro['Beginning page'].iloc[i])
            if pmid:
                neuro.at[i, 'extra_info'] = get_pubmed_abstract(pmid)
            else:
                neuro.at[i, 'extra_info'] = np.nan
        except Exception as e:
            print(i)
            print(f"An unexpected error occurred: {e}")
        time.sleep(1)

neuro[['journal_info','title','authors','author_info', 'extra_comments', 'abstract','doi']] = neuro['extra_info'].apply(split_and_fix_paragraphs).apply(pd.Series)
neuro.to_csv("open_access_journal_data/neuro.csv")

66
128
154
199
209
223
266


In [ ]:
# GENETICS

genetics = df[df['Journal'] == "Genetics"].reset_index(drop = True).copy()

genetics[['journal_info','title','authors','author_info','abstract','doi']] = genetics['extra_info'].apply(split_paragraphs).apply(pd.Series)

import time

for i in range(0, len(genetics )):
    if genetics['title'].iloc[i] is None:
        print(i)
        try: 
            pmid = pubmed_search2(
            journal=genetics['Journal'].iloc[i],
            year= genetics['Year'].iloc[i],
            volume= genetics['Volume'].iloc[i],
            issue= genetics['Issue'].iloc[i],
            page= genetics['Beginning page'].iloc[i])
            if pmid:
                genetics.at[i, 'extra_info'] = get_pubmed_abstract(pmid)
            else:
                genetics.at[i, 'extra_info'] = np.nan
        except Exception as e:
            print(i)
            print(f"An unexpected error occurred: {e}")
        time.sleep(1)
genetics['num_paragraphs'] = genetics['extra_info'].apply(count_paragraphs)
genetics['num_paragraphs'].value_counts()

genetics = genetics[genetics['num_paragraphs'] > 5].copy()
genetics[['journal_info','title','authors','author_info', 'extra_comments', 'abstract','doi']] = genetics['extra_info'].apply(split_and_fix_paragraphs).apply(pd.Series)
genetics.to_csv("open_access_journal_data/genetics.csv")

In [ ]:
# PHYSIO

physio = df[df['Journal'] == "J. Appl. Physiol."].reset_index(drop = True).copy()

physio[['journal_info','title','authors','author_info','abstract','doi']] = physio['extra_info'].apply(split_paragraphs).apply(pd.Series)

import time

for i in range(0, len(physio )):
    if physio['title'].iloc[i] is None:
        print(i)
        try: 
            pmid = pubmed_search2(
            journal=physio['Journal'].iloc[i],
            year= physio['Year'].iloc[i],
            volume= physio['Volume'].iloc[i],
            issue= physio['Issue'].iloc[i],
            page= physio['Beginning page'].iloc[i])
            if pmid:
                physio.at[i, 'extra_info'] = get_pubmed_abstract(pmid)
            else:
                physio.at[i, 'extra_info'] = np.nan
        except Exception as e:
            print(i)
            print(f"An unexpected error occurred: {e}")
        time.sleep(1)
physio['num_paragraphs'] = physio['extra_info'].apply(count_paragraphs)
physio['num_paragraphs'].value_counts()

physio[['journal_info','title','authors','author_info', 'extra_comments', 'abstract','doi']] = physio['extra_info'].apply(split_and_fix_paragraphs).apply(pd.Series)
physio.to_csv("open_access_journal_data/physio.csv")

In [ ]:
# FASEB

faseb = df[df['Journal'] == "Faseb J."].reset_index(drop = True).copy()

faseb[['journal_info','title','authors','author_info','abstract','doi']] = faseb['extra_info'].apply(split_paragraphs).apply(pd.Series)

import time

for i in range(0, len(faseb )):
    if faseb['title'].iloc[i] is None:
        print(i)
        try: 
            pmid = pubmed_search2(
            journal=faseb['Journal'].iloc[i],
            year= faseb['Year'].iloc[i],
            volume= faseb['Volume'].iloc[i],
            issue= faseb['Issue'].iloc[i],
            page= faseb['Beginning page'].iloc[i])
            if pmid:
                faseb.at[i, 'extra_info'] = get_pubmed_abstract(pmid)
            else:
                faseb.at[i, 'extra_info'] = np.nan
        except Exception as e:
            print(i)
            print(f"An unexpected error occurred: {e}")
        time.sleep(1)
faseb['num_paragraphs'] = faseb['extra_info'].apply(count_paragraphs)
faseb['num_paragraphs'].value_counts()

faseb = faseb[faseb['num_paragraphs'] > 5].copy()
faseb[['journal_info','title','authors','author_info', 'extra_comments', 'abstract','doi']] = faseb['extra_info'].apply(split_and_fix_paragraphs).apply(pd.Series)
faseb.to_csv("open_access_journal_data/faseb.csv")